# Week 3 - Activity

## Import Required Libraries

In [21]:
import pandas as pd
import numpy as np

## Load Dataset

In [22]:
telematics_df = pd.read_csv('../data/raw/raw_telematics.csv')
telematics_df.head()

,driver_id,driver_name,date,route_id,vehicle_id,shift,trip_distance,trip_duration,speed,braking_events,harsh_turns
0,D004,Driver_4,2025-01-04,R1,V9,morning,38.4,33,29.8,3,0
1,D019,Driver_19,2025-02-24,R5,V1,night,6.4,22,36.3,0,0
2,D007,Driver_7,2025-03-25,R3,V1,morning,36.6,46,51.4,3,2
3,D014,Driver_14,2025-02-13,R1,V2,afternoon,17.5,31,73.4,0,0
4,D004,Driver_4,2025-02-15,R4,V9,morning,43.1,60,76.5,1,1


## Inspect and Clean Dataset (if needed)

In [23]:
telematics_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   driver_id       500 non-null    str    
 1   driver_name     500 non-null    str    
 2   date            500 non-null    str    
 3   route_id        500 non-null    str    
 4   vehicle_id      500 non-null    str    
 5   shift           500 non-null    str    
 6   trip_distance   500 non-null    float64
 7   trip_duration   500 non-null    int64  
 8   speed           500 non-null    float64
 9   braking_events  500 non-null    int64  
 10  harsh_turns     500 non-null    int64  
dtypes: float64(2), int64(3), str(6)
memory usage: 43.1 KB


In [24]:
telematics_df.describe()

,trip_distance,trip_duration,speed,braking_events,harsh_turns
count,500.000000,500.000000,500.000000,500.000000,500.00000
mean,27.801800,52.084000,55.746000,1.434000,1.02000
std,12.547753,21.479471,19.961964,1.223371,1.03815
min,5.000000,15.000000,20.700000,0.000000,0.00000
25%,17.400000,34.000000,38.600000,0.000000,0.00000
50%,27.950000,51.500000,55.350000,1.000000,1.00000
75%,38.000000,71.000000,73.425000,2.000000,2.00000
max,50.000000,89.000000,89.700000,5.000000,5.00000


### Missing Values

In [25]:
print('Missing Values Count:')
telematics_df.isna().sum()

Missing Values Count:


driver_id         0
driver_name       0
date              0
route_id          0
vehicle_id        0
shift             0
trip_distance     0
trip_duration     0
speed             0
braking_events    0
harsh_turns       0
dtype: int64

### Duplicated Rows

In [26]:
print(f'Duplicated Rows: {telematics_df.duplicated().sum()}')

Duplicated Rows: 0


## Feature Engineer Columns 

**Trip Delays (delays_minutes) :**
* This is the total time in minutes that a trip exceeded its expected duration due to inefficiencies.
* It is calculated by subtracting the **expected travel time** (distance divided by speed, multiplied by 60) from the actual **trip duration**. Decimal values are converted to integers for consistency.
* It measures trip inefficiency and assumed to be used as a direct penalty against the driver's final rating.

In [28]:
expected_time = np.where(
    telematics_df['speed'] > 0, 
    (telematics_df['trip_distance'] / telematics_df['speed']) * 60, 
    0 
)

delays_minute = (telematics_df['trip_duration'] - expected_time).astype(int).clip(lower=0)
telematics_df['delays_minutes'] = delays_minute

telematics_df['delays_minutes'].head()

0     0
1    11
2     3
3    16
4    26
Name: delays_minutes, dtype: int64

**Behavioral Problems :** 
* This is a metric assumed to capture abrupt deceleration and aggressive driving maneuvers.
* It is represented directly by the total count of **harsh braking events** recorded during the trip.
* This metrics assumes high counts indicate tailgating, distracted driving, or poor situational awareness.

In [31]:
telematics_df['behavioral_problems'] = telematics_df['braking_events']

telematics_df['behavioral_problems'].head()

0    3
1    0
2    3
3    0
4    1
Name: behavioral_problems, dtype: int64

**Traffic & Policy Violations (speeding and violations) :**
* This is an aggregate count of unsafe driving actions that violate road safety rules or company policies.
* It is represented by the sum of binary **speeding flags** (1 if vehicle speed exceeds 60 km/h, else 0) and the **total count of harsh turning events** .
* This measures a driver’s non-compliance with speed limits and controlled vehicle handling.

In [35]:
telematics_df['speeding'] = (telematics_df['speed'] > 60).astype(int)

telematics_df['speeding'].head()

0    0
1    0
2    0
3    1
4    1
Name: speeding, dtype: int64

In [36]:
telematics_df['violations'] = telematics_df['speeding'] + telematics_df['harsh_turns']

telematics_df['violations'].head()

0    0
1    0
2    2
3    1
4    2
Name: violations, dtype: int64

**Aggressive Incident (aggressive_incident) :** 
* It is a high-risk driving flag indicating near-miss situations or extreme reckless behavior.
* It is represented by a binary flag (1 or 0) triggered only when a trip exhibits a dangerous combination of **high frequency braking** (≥ 3 braking events) while traveling at **high speeds** (> 70 km/h).
* It identifies outlier trips where severe safety interventions may be required.

In [38]:
telematics_df['aggressive_incident'] = ((telematics_df['braking_events'] >= 3) & (telematics_df['speed'] > 70)).astype(int)

telematics_df['aggressive_incident'].head()

0    0
1    0
2    0
3    0
4    0
Name: aggressive_incidents, dtype: int64

**Driver Safety Score (driver_rating) :**
* This is an assumed performance metric evaluating overall driver safety and efficiency on a scale from 1.0 (Poor) to 5.0 (Excellent).
* The base score starts at 5.0 and applies deductions for operational delays, behavioral events, safety violations, and aggressive incidents.
* **Deduction Weights (weights are just assumed and not based on an actual computation):**
    * **Trip Delays:** -0.03 per minute
    * **Behavioral Problems:** -0.20 per event
    * **Violations:** -0.15 per event
    * **Aggressive Incident:** -0.50  if considered aggressive incident

In [40]:
rating = 5.0
rating -= telematics_df['delays_minutes'] * 0.03
rating -= telematics_df['behavioral_problems'] * 0.2
rating -= telematics_df['violations'] * 0.15
rating -= telematics_df['aggressive_incident'] * 0.5

telematics_df['driver_rating'] = rating.clip(lower=1.0, upper=5.0).round(2)
telematics_df['driver_rating'].head()

0    4.40
1    4.67
2    4.01
3    4.37
4    3.72
Name: driver_rating, dtype: float64

## Final Dataset

In [46]:
telematics_df.head()

,driver_id,driver_name,date,route_id,vehicle_id,shift,trip_distance,trip_duration,speed,braking_events,harsh_turns,delays_minutes,behavioral_problems,speeding,violations,aggressive_incidents,driver_rating
0,D004,Driver_4,2025-01-04,R1,V9,morning,38.4,33,29.8,3,0,0,3,0,0,0,4.40
1,D019,Driver_19,2025-02-24,R5,V1,night,6.4,22,36.3,0,0,11,0,0,0,0,4.67
2,D007,Driver_7,2025-03-25,R3,V1,morning,36.6,46,51.4,3,2,3,3,0,2,0,4.01
3,D014,Driver_14,2025-02-13,R1,V2,afternoon,17.5,31,73.4,0,0,16,0,1,1,0,4.37
4,D004,Driver_4,2025-02-15,R4,V9,morning,43.1,60,76.5,1,1,26,1,1,2,0,3.72


In [51]:
telematics_df.to_csv("../data/cleaned/driver_profiles.csv", index=False)
print(f"Dataset \'driver_profiles.csv\' saved: {len(telematics_df)} rows")

Dataset 'driver_profiles.csv' saved: 500 rows
